# GRIFFIN Workday Scrape -- Training Data Collection

> **W&M HR Position Classification & Pay Matching Tool**
> Created: March 30, 2026 | Kernel: `griffin` (conda)

This notebook collects, parses, censors, and validates W&M Workday job postings
to produce a labeled training dataset for the GRIFFIN classification pipeline.

| Step | What it does | Output |
|------|-------------|--------|
| 1. Setup | Import packages, define paths and constants | -- |
| 2. List Pagination | Hit the Workday list API, paginate all pages, filter faculty | `raw_workday/list_page_*.json`, `staff_postings.json` |
| 3. Detail Fetching | Fetch each staff posting's full detail JSON | `raw_workday/detail/*.json` |
| 4. Parse + Censor | Extract labels from HTML, censor classification-revealing text | `workday_postings.csv`, `workday_training.csv` |
| 5. Validation | Programmatic leakage scan + 15 visual spot-checks | PASS/FAIL verdict |
| 6. What's Next | Feature engineering, H2O AutoML, LangChain multi-agent | -- |

### Dataset Summary

| Metric | Value |
|--------|-------|
| Total Workday postings | 150 |
| Staff retained | 103 |
| Faculty excluded | 47 |
| Labels extracted | comp_grade, job_profile_code, job_family, exempt_status, pay_type |
| Leakage scan result | 0 leaks across 7 patterns |

### How this notebook works

Every code cell is **idempotent** -- running the notebook a second time loads
from cached files on disk and never re-fetches from the Workday API. This is
both polite (rate limiting) and practical (the API data is a point-in-time
snapshot, so re-fetching would give different results as postings change).

**Insight:** This notebook is part of the GRIFFIN project pipeline. The previous
notebook (`griffin_environment_setup.ipynb`) created the conda environment.
The next notebook will engineer tabular features from the censored text for
H2O AutoML training.

---
## 1. Setup + Imports

We need a focused set of packages for web scraping, HTML parsing, and data export.
All of these were installed by `griffin_environment_setup.ipynb`.

| Package | Import | Purpose |
|---------|--------|---------|
| `requests` | `requests` | HTTP client for the Workday JSON API (POST for list, GET for detail) |
| `json` | `json` | Parse and cache API responses as JSON files |
| `os` | `os` | File path construction and existence checks |
| `time` | `time` | Rate limiting between API requests (1.5s delay) |
| `re` | `re` | Regex patterns for label extraction and censoring |
| `math` | `math` | `ceil()` for computing expected page count |
| `csv` | `csv` | Write faculty exclusion log |
| `glob` | `glob` | List cached detail files by pattern |
| `html` | `html` | Decode HTML entities (`&amp;` to `&`) in job_family values |
| `random` | `random` | Reproducible spot-check sampling (seed=42) |
| `BeautifulSoup` | `bs4` | Parse Workday HTML job descriptions into clean plain text |
| `pandas` | `pandas` | DataFrame construction and CSV export |

**Why these specific packages?** The Workday job site exposes a JSON API
(no Selenium or browser automation needed), but the `jobDescription` field
contains raw HTML that needs parsing. BeautifulSoup handles the HTML-to-text
conversion, while regex handles the structured label extraction.

In [ ]:
import requests
import json
import os
import time
import re
import math
import csv
import glob
import html as html_module  # avoid collision with variable names
import random
from bs4 import BeautifulSoup
import pandas as pd

print(f"pandas {pd.__version__}")
print(f"requests {requests.__version__}")
print("All imports OK.")

### Constants and Paths

The Workday API for W&M uses a two-endpoint pattern:

1. **List endpoint** (POST): Returns paginated job summaries (title, location,
   bulletFields). We POST `{"limit": 20, "offset": N}` to page through all results.
2. **Detail endpoint** (GET): Returns the full posting for a single job, including
   the HTML `jobDescription` that contains our classification labels.

We cache everything to `raw_workday/` so the notebook is rerunnable without
hitting the API again.

In [ ]:
# --- Project paths ---
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))  # notebook's directory
PROJECT_ROOT = os.path.join(NOTEBOOK_DIR, "..")
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "cache", "raw_workday")
DETAIL_DIR = os.path.join(RAW_DIR, "detail")
STAFF_FILE = os.path.join(RAW_DIR, "staff_postings.json")

# Output CSVs
OUT_POSTINGS = os.path.join(PROJECT_ROOT, "data", "training", "workday_postings.csv")
OUT_TRAINING = os.path.join(PROJECT_ROOT, "data", "training", "workday_training.csv")
OUT_EXCLUSIONS = os.path.join(PROJECT_ROOT, "data", "training", "workday_exclusions.csv")

# --- API configuration ---
API_URL = "https://williammary.wd12.myworkdayjobs.com/wday/cxs/williammary/WM/jobs"
BASE_URL = "https://williammary.wd12.myworkdayjobs.com/wday/cxs/williammary/WM"
HEADERS_POST = {"Content-Type": "application/json", "Accept": "application/json"}
HEADERS_GET = {"Accept": "application/json"}
LIMIT = 20           # Workday's page size
SLEEP_SECONDS = 1.5  # Polite rate limit between uncached API calls

# --- Create cache directories ---
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DETAIL_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw cache:    {RAW_DIR}")
print(f"Detail cache: {DETAIL_DIR}")

### Helper: Em-Dash Sanitization

The Workday API occasionally returns em-dashes (U+2014) and en-dashes (U+2013)
in job titles and descriptions. These cause encoding issues in CSV exports and
make text inconsistent across platforms. We replace them with `--` throughout
the entire pipeline -- from raw caching through final export.

In [ ]:
def sanitize_text(text):
    """Replace em-dashes and en-dashes with double hyphens."""
    if isinstance(text, str):
        return text.replace("\u2014", "--").replace("\u2013", "--")
    return text


def sanitize_obj(obj):
    """Recursively sanitize all strings in a dict/list."""
    if isinstance(obj, str):
        return sanitize_text(obj)
    elif isinstance(obj, list):
        return [sanitize_obj(item) for item in obj]
    elif isinstance(obj, dict):
        return {sanitize_obj(k): sanitize_obj(v) for k, v in obj.items()}
    return obj


print("Helpers loaded.")

---
## 2. List Pagination

### How the Workday List API works

W&M's Workday job site at `williammary.wd12.myworkdayjobs.com` exposes a JSON
API behind the scenes. When you visit the careers page and scroll, your browser
sends POST requests to the `/jobs` endpoint with pagination parameters:

```json
{"limit": 20, "offset": 0, "appliedFacets": {}, "searchText": ""}
```

The response includes:
- `total`: The total number of postings (150 at time of collection)
- `jobPostings`: An array of up to 20 posting summaries
- Each posting has `title`, `externalPath`, `bulletFields`, `locationsText`, `postedOn`

We paginate through all pages (offset 0, 20, 40, ... 140), cache each response
to `raw_workday/list_page_{offset}.json`, and then filter out faculty postings.

### robots.txt and rate limiting

The Workday `robots.txt` does not disallow the `/wday/cxs/` API path. We add
a 1.5-second delay between uncached requests out of courtesy. The total data
volume is small (150 postings across 8 pages) -- this is a one-time collection,
not continuous scraping.

In [ ]:
# --- Paginate and cache all list pages ---
offset = 0
total = None
all_postings = []
pages_fetched = 0
pages_cached = 0

while True:
    filename = f"list_page_{offset}.json"
    filepath = os.path.join(RAW_DIR, filename)

    if os.path.exists(filepath):
        # Load from cache -- never re-fetch
        with open(filepath, "r", encoding="utf-8") as f:
            page_data = json.load(f)
        count = len(page_data.get("jobPostings", []))
        print(f"  Loaded from cache: {filename} ({count} postings)")
        pages_cached += 1
    else:
        # Fetch from API
        body = {"limit": LIMIT, "offset": offset, "appliedFacets": {}, "searchText": ""}
        print(f"  Fetching page offset={offset}...", end=" ")
        resp = requests.post(API_URL, headers=HEADERS_POST, json=body)
        resp.raise_for_status()
        page_data = sanitize_obj(resp.json())
        count = len(page_data.get("jobPostings", []))
        print(f"{count} postings")

        # Cache to disk
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(page_data, f, indent=2, ensure_ascii=False)
        pages_fetched += 1
        time.sleep(SLEEP_SECONDS)

    # Track total from first response
    if total is None:
        total = page_data.get("total", 0)
        expected_pages = math.ceil(total / LIMIT)
        print(f"\n  API reports total={total}, expecting {expected_pages} pages\n")

    all_postings.extend(page_data.get("jobPostings", []))
    offset += LIMIT

    if offset >= total:
        break

print(f"\nPagination complete: {pages_fetched} fetched, {pages_cached} from cache")
print(f"Total postings collected: {len(all_postings)}")

### Filter Faculty Postings

GRIFFIN classifies **staff** positions only. Faculty positions have a completely
different pay and classification structure (academic ranks, tenure tracks) that
does not map to the DHRM career group taxonomy we use for staff.

We identify faculty by checking `bulletFields` for the word "Faculty"
(case-insensitive). This field contains the job family, time type, and
worker sub-type -- if any bullet mentions faculty, we exclude the posting.
Exclusions are logged to `workday_exclusions.csv` for audit trail.

In [ ]:
# --- Filter faculty vs staff ---
staff_postings = []
faculty_postings = []

for job in all_postings:
    bullet_fields = job.get("bulletFields", [])
    is_faculty = any("faculty" in sanitize_text(str(bf)).lower() for bf in bullet_fields)
    if is_faculty:
        faculty_postings.append(job)
    else:
        staff_postings.append(job)

assert len(staff_postings) + len(faculty_postings) == len(all_postings), "Count mismatch!"

print(f"Total: {len(all_postings)}")
print(f"Staff (retained): {len(staff_postings)}")
print(f"Faculty (excluded): {len(faculty_postings)}")

# --- Log faculty exclusions ---
with open(OUT_EXCLUSIONS, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "externalPath", "job_family", "reason"])
    writer.writeheader()
    for job in faculty_postings:
        bullet_fields = job.get("bulletFields", [])
        faculty_fields = [sanitize_text(str(bf)) for bf in bullet_fields
                          if "faculty" in str(bf).lower()]
        job_family = faculty_fields[0] if faculty_fields else "unknown"
        writer.writerow({
            "title": sanitize_text(job.get("title", "")),
            "externalPath": sanitize_text(job.get("externalPath", "")),
            "job_family": sanitize_text(job_family),
            "reason": "bulletFields contains faculty (case-insensitive)"
        })

print(f"\nWrote {len(faculty_postings)} exclusions to workday_exclusions.csv")

# --- Save staff postings for detail fetching ---
staff_output = []
for job in staff_postings:
    staff_output.append({
        "title": sanitize_text(job.get("title", "")),
        "externalPath": sanitize_text(job.get("externalPath", "")),
        "bulletFields": [sanitize_text(str(bf)) for bf in job.get("bulletFields", [])],
        "locationsText": sanitize_text(job.get("locationsText", "")),
        "postedOn": sanitize_text(job.get("postedOn", ""))
    })

with open(STAFF_FILE, "w", encoding="utf-8") as f:
    json.dump(staff_output, f, indent=2, ensure_ascii=False)

print(f"Wrote {len(staff_output)} staff postings to raw_workday/staff_postings.json")

---
## 3. Detail Fetching

### Why we need the detail endpoint

The list endpoint gives us summaries (title, location, bulletFields), but the
classification labels we need -- compensation grade, job profile code, job family,
FLSA status -- are buried inside the `jobDescription` HTML on the detail page.

Each staff posting has an `externalPath` like `/job/William--Mary/Assistant-Controller_JR101404`.
We append this to the base URL and GET the full detail JSON. The response contains:

```
{
  "jobPostingInfo": {
    "title": "...",
    "jobDescription": "<p>...HTML with embedded labels...</p>",
    "jobReqId": "JR101404",
    "location": "...",
    ...
  },
  "hiringOrganization": {...},
  ...
}
```

### Caching strategy

Each detail response is cached to `raw_workday/detail/{sanitized_path}.json`.
The filename replaces `/` with `_` from the `externalPath`. On subsequent runs,
cached files are loaded directly -- no API call, no sleep delay.

In [ ]:
def sanitize_path(external_path):
    """Replace / with _ and strip leading underscore for safe filenames."""
    return external_path.strip("/").replace("/", "_")


# --- Fetch all detail pages ---
with open(STAFF_FILE, "r", encoding="utf-8") as f:
    postings_list = json.load(f)

n_total = len(postings_list)
n_fetched = 0
n_cached = 0

print(f"Starting detail fetch for {n_total} staff postings...\n")

for i, posting in enumerate(postings_list, 1):
    ext_path = posting["externalPath"]
    safe_name = sanitize_path(ext_path)
    cache_file = os.path.join(DETAIL_DIR, f"{safe_name}.json")

    if os.path.exists(cache_file):
        n_cached += 1
    else:
        url = f"{BASE_URL}{ext_path}"
        resp = requests.get(url, headers=HEADERS_GET, timeout=30)
        resp.raise_for_status()
        data = sanitize_obj(resp.json())
        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        n_fetched += 1
        time.sleep(SLEEP_SECONDS)

    if i % 20 == 0 or i == n_total:
        print(f"  Progress: {i}/{n_total} ({n_cached} cached, {n_fetched} fetched)")

print(f"\nDone. {n_fetched} fetched, {n_cached} from cache. Total: {n_fetched + n_cached}")

### Structure Inspection

Before writing any parsing code, we inspect 3 sample detail JSONs to understand
where the classification labels live. This is a critical step -- parsing HTML
without understanding the structure leads to brittle regex and missed fields.

**Key findings from inspection:**

| Label | Location | Format |
|-------|----------|--------|
| Compensation Grade | `<b>Compensation Grade:</b></p>S14` in jobDescription HTML | `S##` or `H##` |
| Job Profile | `<b>Job Profile:</b></p>JP0591 - Title - Exempt - Salary - S14` | Compound string with JP code |
| Job Family | `<b>Job Family:</b></p>Staff - Financial Analysis` | Plain text after label |
| Job Requisition ID | `<b>Job Requisition:</b></p>JR101351 Title (Open)` | `JR######` at start of value |
| FLSA Status | Embedded in Job Profile compound string | "Exempt" or "Nonexempt" |
| Pay Type | Embedded in Job Profile compound string | "Salary" or "Hourly" |
| Department | `<b>Department:</b></p>CC00456 WM001 \| WMUO \| Budget & Financial Planning` | Full department path |

The HTML follows a consistent `<p><b>Label:</b></p>Value` pattern across all
103 postings. This makes regex extraction reliable.

In [ ]:
# --- Quick structure check: inspect 3 sample detail files ---
detail_files = sorted(os.listdir(DETAIL_DIR))
sample_indices = [0, len(detail_files) // 2, len(detail_files) - 1]

for idx in sample_indices:
    fname = detail_files[idx]
    fpath = os.path.join(DETAIL_DIR, fname)
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    jpi = data.get("jobPostingInfo", {})
    jd = jpi.get("jobDescription", "")

    print(f"--- {fname} ---")
    print(f"  Top-level keys: {list(data.keys())}")
    print(f"  jobPostingInfo keys: {list(jpi.keys())}")
    print(f"  jobDescription length: {len(jd)} chars")

    # Show label locations
    comp_grades = re.findall(r'\b[SH]\d{2}\b', json.dumps(data))
    jp_codes = re.findall(r'\bJP\d{4,}\b', json.dumps(data))
    print(f"  Comp grades found: {comp_grades[:4]}")
    print(f"  JP codes found: {jp_codes[:4]}")
    print()

---
## 4. Parse + Label Extract + Censor

### Label extraction strategy

We extract 7 fields from each posting's HTML `jobDescription`:

1. **Job Requisition ID** (`JR######`) -- from the "Job Requisition:" label or structured `jobReqId`
2. **Job Family** -- from the "Job Family:" label (e.g., "Staff - Financial Analysis")
3. **Department** -- from the "Department:" label
4. **Compensation Grade** -- `S##` or `H##` from "Compensation Grade:" label
5. **Job Profile Code** -- `JP####` parsed from the "Job Profile:" compound string
6. **Exempt Status** -- "Exempt" or "Nonexempt" from the Job Profile string or body text
7. **Pay Type** -- "Salary" or "Hourly" from the Job Profile string or "Pay Rate Type:" label

The extraction uses a common pattern: find `<b>Label:</b></p>` in the HTML, then
capture everything until the next `<p` or `<div` tag.

### Why we censor the text

> **Insight:** The ML model we train later (H2O AutoML) must learn to predict
> classification from the *content* of the position description -- duties,
> qualifications, supervision scope. If the training text contains the answer
> (e.g., "Compensation Grade: S14" or "Job Family: Staff - Financial Analysis"),
> the model will memorize these strings instead of learning the underlying
> patterns. This is **data leakage** -- the #1 cause of models that look
> perfect in training but fail completely in production.
>
> We censor by removing: all label headers and values, Job Profile codes,
> compensation grades, JR IDs, dollar amounts (salary ranges), job family
> values, FLSA status lines, and EEO boilerplate. What remains is the
> substantive job description that a human HR analyst would read.

In [ ]:
# ===================================================================
# Label extraction helpers (operate on raw HTML)
# ===================================================================

def extract_label_value(html_text, label):
    """
    Extract the plain-text value that follows a bold label in Workday HTML.
    Pattern: <b>Label:</b></p>VALUE<p  or  <b>Label:</b></p>VALUE<
    Returns None if label not found.
    """
    pattern = re.compile(
        r'<b>\s*' + re.escape(label) + r'\s*</b>\s*</p>\s*(.*?)(?=<p|<div|$)',
        re.IGNORECASE | re.DOTALL
    )
    m = pattern.search(html_text)
    if m:
        val = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        if val:
            return sanitize_text(val)
    return None


def extract_job_req_id(html_text, structured_id):
    """Extract JR###### from HTML or use structured field as fallback."""
    val = extract_label_value(html_text, "Job Requisition:")
    if val:
        m = re.search(r'(JR\d{5,6})', val)
        if m:
            return m.group(1)
    return structured_id if structured_id else None


def extract_job_family(html_text):
    """Extract Job Family from the bold label."""
    val = extract_label_value(html_text, "Job Family:")
    if val:
        # Decode HTML entities: &amp; -> &
        val = html_module.unescape(val)
    return val


def extract_department(html_text):
    """Extract Department from the bold label."""
    return extract_label_value(html_text, "Department:")


def extract_compensation_grade(html_text):
    """Extract S## or H## compensation grade."""
    val = extract_label_value(html_text, "Compensation Grade:")
    if val:
        m = re.search(r'([SH]\d{1,2})', val)
        if m:
            return m.group(1)
    return None


def extract_job_profile_full(html_text):
    """Extract the full Job Profile compound string."""
    return extract_label_value(html_text, "Job Profile:")


def extract_job_profile_code(profile_str):
    """Parse JP#### from the job profile string."""
    if not profile_str:
        return None
    m = re.search(r'(JP\d{3,6})', profile_str)
    return m.group(1) if m else None


def extract_exempt_status(profile_str, html_text):
    """Parse Exempt/Nonexempt from job profile string, with HTML fallbacks."""
    if profile_str:
        if re.search(r'\bNon-?exempt\b', profile_str, re.IGNORECASE):
            return "Nonexempt"
        if re.search(r'\bExempt\b', profile_str, re.IGNORECASE):
            return "Exempt"
    # Fallback: check body text
    if re.search(r'non-?exempt\s+position', html_text, re.IGNORECASE):
        return "Nonexempt"
    if re.search(r'\bexempt\s+position\b', html_text, re.IGNORECASE):
        return "Exempt"
    m = re.search(r'FLSA\s*:?\s*(Non-?exempt|Exempt)', html_text, re.IGNORECASE)
    if m:
        return "Nonexempt" if re.match(r'non', m.group(1), re.IGNORECASE) else "Exempt"
    return None


def extract_pay_type(profile_str, html_text):
    """Parse Salary/Hourly from job profile string, with HTML fallbacks."""
    if profile_str:
        if re.search(r'\bHourly\b', profile_str, re.IGNORECASE):
            return "Hourly"
        if re.search(r'\bSalary\b', profile_str, re.IGNORECASE):
            return "Salary"
    val = extract_label_value(html_text, "Pay Rate Type:")
    if val:
        if re.search(r'Hourly', val, re.IGNORECASE):
            return "Hourly"
        if re.search(r'Salary', val, re.IGNORECASE):
            return "Salary"
    if re.search(r'<b>\s*Hourly Rate\s*:?\s*</b>', html_text, re.IGNORECASE):
        return "Hourly"
    if re.search(r'<b>\s*Period Salary Plan\s*:?\s*</b>', html_text, re.IGNORECASE):
        return "Salary"
    if re.search(r'<b>\s*Annual Salary\s*:?\s*</b>', html_text, re.IGNORECASE):
        return "Salary"
    return None


print("Label extraction helpers loaded (7 extractors).")

### HTML-to-Text Conversion and Censoring

The censoring step has two phases:

1. **HTML to plain text**: BeautifulSoup parses the HTML tags, replacing `<br>` with
   newlines, `<li>` with bullet points, and `<p>` with paragraph breaks.
2. **Line-by-line censoring**: Each line is checked against 30+ skip patterns that
   catch label headers, standalone values, salary ranges, EEO boilerplate, and
   Workday URLs. Lines that survive are cleaned of inline JP codes, JR codes,
   compensation grade tokens, and dollar amounts.

The result is clean, readable text that a human could use to classify the
position -- but without any of the structured classification metadata that
would let the ML model cheat.

In [ ]:
# ===================================================================
# HTML-to-text and censoring
# ===================================================================

def html_to_plain_text(html_text):
    """Convert HTML to readable plain text using BeautifulSoup."""
    soup = BeautifulSoup(html_text, "html.parser")
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for li in soup.find_all("li"):
        li.insert(0, "- ")
        li.append("\n")
    for p in soup.find_all("p"):
        p.append("\n")
    return soup.get_text()


def censor_text(plain_text, labels):
    """
    Remove classification-revealing content from plain text.
    labels dict has: job_req_id, job_family, job_profile_full,
    compensation_grade, job_profile_code, exempt_status, pay_type
    """
    lines = plain_text.split("\n")
    censored_lines = []

    # --- Build skip patterns (entire-line removal) ---
    skip_patterns = [
        # Structural label headers
        re.compile(r'^\s*JR\d{5,6}\b', re.IGNORECASE),
        re.compile(r'^\s*Job Requisition\s*:', re.IGNORECASE),
        re.compile(r'^\s*JP\d{3,6}\b'),
        re.compile(r'^\s*Job Profile\s*:', re.IGNORECASE),
        re.compile(r'^\s*Compensation Grade\s*:', re.IGNORECASE),
        re.compile(r'^\s*[SH]\d{1,2}\s*$'),
        re.compile(r'^\s*Job Family\s*:', re.IGNORECASE),
        re.compile(r'^\s*FLSA\s*:', re.IGNORECASE),
        re.compile(r'^\s*Worker Sub[\s-]?Type\s*:', re.IGNORECASE),
        re.compile(r'^\s*Pay Rate Type\s*:', re.IGNORECASE),
        re.compile(r'^\s*Period Salary Plan\s*:', re.IGNORECASE),
        re.compile(r'^\s*Job Posting Title\s*:', re.IGNORECASE),
        re.compile(r'^\s*Job Requisition Primary Location\s*:', re.IGNORECASE),
        re.compile(r'^\s*Primary Job Posting Location\s*:', re.IGNORECASE),
        re.compile(r'^\s*Additional Job Description\s*:?\s*$', re.IGNORECASE),
        re.compile(r'^\s*Qualifications\s*:\s*$', re.IGNORECASE),
        re.compile(r'^\s*Recruiting Start Date\s*:', re.IGNORECASE),
        re.compile(r'^\s*Review Date\s*:', re.IGNORECASE),
        re.compile(r'^\s*Position Restrictions\s*:', re.IGNORECASE),
        re.compile(r'^\s*\d{4}-\d{2}-\d{2}\s*$'),
        re.compile(r'^\s*Number of Openings\s*:', re.IGNORECASE),
        re.compile(r'^\s*Remote Eligibility\s*:', re.IGNORECASE),
        re.compile(r'\bBand\s+\d+', re.IGNORECASE),
        re.compile(r'\bPay\s+Band\b', re.IGNORECASE),
        # Standalone value lines (orphans after header removal)
        re.compile(r'^\s*Staff\s*-\s*[A-Z][\w\s&,;-]+\s*$', re.IGNORECASE),
        re.compile(
            r'^\s*(Regular \(benefited\)|Wage-Temporary[^$]*|Regular \(non-benefited\))\s*$',
            re.IGNORECASE
        ),
        re.compile(r'^\s*(Hourly|Salary)\s*$', re.IGNORECASE),
        re.compile(r'^\s*BS\s*$'),
        re.compile(r'^\s*(Part time|Full time)\s*$', re.IGNORECASE),
        # EEO boilerplate and noise
        re.compile(r'EEO is the Law', re.IGNORECASE),
        re.compile(r'Background Check\s*:', re.IGNORECASE),
        re.compile(r'Remote Work Disclaimer\s*:', re.IGNORECASE),
        re.compile(r'www\.eeoc\.gov', re.IGNORECASE),
        re.compile(r'Know Your Rights', re.IGNORECASE),
        re.compile(r'myworkdayjobs\.com', re.IGNORECASE),
    ]

    # Add job-family-specific value pattern if known
    if labels.get("job_family"):
        jf_decoded = html_module.unescape(labels["job_family"])
        jf_pattern = re.escape(jf_decoded)
        skip_patterns.append(re.compile(r'^\s*' + jf_pattern + r'\s*$', re.IGNORECASE))

    # Salary line patterns (entire line removed)
    salary_line_pattern = re.compile(
        r'^\s*(Posting [Rr]ange|Hiring [Rr]ange|Salary|Annual Salary|'
        r'Hourly Rate|SALARY|SIGN-ON BONUS)\s*:',
        re.IGNORECASE
    )

    # Inline removal patterns
    dollar_pattern = re.compile(
        r'\$\s*[\d,]+(?:\.\d{2})?(?:\s*[-to]+\s*\$\s*[\d,]+(?:\.\d{2})?)?'
    )

    for line in lines:
        stripped = line.strip()
        if not stripped:
            censored_lines.append("")
            continue

        # Check skip patterns
        skip = False
        for pat in skip_patterns:
            if pat.search(stripped):
                skip = True
                break
        if skip:
            continue

        if salary_line_pattern.search(stripped):
            continue

        # Inline removal
        cleaned = dollar_pattern.sub("", line)
        cleaned = re.sub(r'\bJP\d{3,6}\b', '', cleaned)
        cleaned = re.sub(r'(?<![A-Za-z])JR\d{5,6}', '', cleaned)
        cleaned = re.sub(r'\b[SH]\d{1,2}\b', '', cleaned)
        cleaned = re.sub(r'^\s*-?\s*(Non-?[Ee]xempt|Exempt)\s*-?\s*$', '', cleaned)
        cleaned = re.sub(r'\s*-\s*-\s*-\s*', ' - ', cleaned)
        cleaned = re.sub(r'\s{2,}', ' ', cleaned)
        cleaned = cleaned.strip()

        if cleaned:
            censored_lines.append(cleaned)

    result = "\n".join(censored_lines)
    result = re.sub(r'\n{3,}', '\n\n', result)
    return result.strip()


print("Censoring engine loaded.")

### Process All 103 Detail JSONs

Now we run the full pipeline on every cached detail file:
1. Load the detail JSON
2. Extract all 7 labels from the HTML
3. Convert HTML to plain text
4. Censor the plain text
5. Collect into a DataFrame

In [ ]:
# --- Load staff postings list (for externalPath mapping) ---
with open(STAFF_FILE) as f:
    staff_list = json.load(f)

path_by_jr = {}
for sp in staff_list:
    m = re.search(r'(JR\d{5,6})', sp.get("externalPath", ""))
    if m:
        path_by_jr[m.group(1)] = sp["externalPath"]

# --- Process each detail JSON ---
records = []
detail_file_list = sorted(glob.glob(os.path.join(DETAIL_DIR, "*.json")))
print(f"Processing {len(detail_file_list)} detail files...\n")

for fpath in detail_file_list:
    with open(fpath) as f:
        data = json.load(f)

    info = data.get("jobPostingInfo", {})
    html_content = info.get("jobDescription", "")

    # Structured fields (outside HTML)
    title = sanitize_text(info.get("title", ""))
    location = sanitize_text(info.get("location", ""))
    job_req_id_structured = info.get("jobReqId", "")
    external_url = info.get("externalUrl", "")

    # Derive external_path
    external_path = ""
    if external_url:
        m = re.search(r'/WM(/job/.+)', external_url)
        if m:
            external_path = m.group(1)
    if not external_path and job_req_id_structured:
        external_path = path_by_jr.get(job_req_id_structured, "")

    # Extract labels from HTML
    job_req_id = extract_job_req_id(html_content, job_req_id_structured)
    job_family = extract_job_family(html_content)
    department = extract_department(html_content)
    compensation_grade = extract_compensation_grade(html_content)
    job_profile_full = extract_job_profile_full(html_content)
    job_profile_code = extract_job_profile_code(job_profile_full)
    exempt_status = extract_exempt_status(job_profile_full, html_content)
    pay_type = extract_pay_type(job_profile_full, html_content)

    # Convert and censor
    raw_text = sanitize_text(html_to_plain_text(html_content))
    labels_dict = {
        "job_req_id": job_req_id,
        "job_family": job_family,
        "job_profile_full": job_profile_full,
        "job_profile_code": job_profile_code,
        "compensation_grade": compensation_grade,
        "exempt_status": exempt_status,
        "pay_type": pay_type,
    }
    censored_text = sanitize_text(censor_text(raw_text, labels_dict))

    records.append({
        "title": title,
        "external_path": external_path,
        "raw_text": raw_text,
        "censored_text": censored_text,
        "compensation_grade": compensation_grade,
        "job_profile_code": job_profile_code,
        "job_family": job_family,
        "exempt_status": exempt_status,
        "pay_type": pay_type,
        "job_req_id": job_req_id,
        "department": department,
        "location": location,
    })

df = pd.DataFrame(records)

# Em-dash guard on all string columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(lambda x: sanitize_text(x) if isinstance(x, str) else x)

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

### Label Coverage Statistics

How many postings have each label extracted? Coverage below 80% would indicate
a parsing issue. We expect near-100% for compensation grade (since every posting
has a salary structure) and slightly lower for fields like job_profile_code
(some postings use a different format).

In [ ]:
print("=" * 60)
print("LABEL COVERAGE")
print("=" * 60)
n = len(df)
for col in ["compensation_grade", "job_profile_code", "job_family",
            "exempt_status", "pay_type", "department"]:
    count = df[col].notna().sum()
    pct = count / n * 100
    print(f"  {col:<24s} {count:>3d}/{n}  ({pct:.1f}%)")

ct_lens = df["censored_text"].str.len()
print(f"\nCensored text length: min={ct_lens.min()}, max={ct_lens.max()}, mean={ct_lens.mean():.0f}")

print(f"\n--- Compensation Grade distribution ---")
print(df["compensation_grade"].value_counts().to_string())

print(f"\n--- Job Family distribution (top 10) ---")
print(df["job_family"].value_counts().head(10).to_string())

### Export CSVs

We produce two output files:

1. **`workday_postings.csv`** (12 columns) -- Full dataset with raw text, censored text,
   and all labels. This is the master file for auditing and inspection.
2. **`workday_training.csv`** (6 columns) -- ML training set with censored text and labels
   only. Raw text is deliberately excluded to prevent accidental leakage during training.

Both use `utf-8-sig` encoding (BOM prefix) so they open correctly in Excel.

In [ ]:
# --- Decode HTML entities in all string columns ---
# Workday HTML contains &amp; which persists after BS4 extraction.
# We need clean text for both ML training and human readability.
import html as _html
for _col in df.select_dtypes(include='object').columns:
    _mask = df[_col].notna()
    df.loc[_mask, _col] = df.loc[_mask, _col].apply(lambda x: _html.unescape(str(x)))

# --- Export workday_postings.csv (full dataset) ---
postings_cols = [
    "title", "external_path", "raw_text", "censored_text",
    "compensation_grade", "job_profile_code", "job_family",
    "exempt_status", "pay_type", "job_req_id", "department", "location"
]
df[postings_cols].to_csv(OUT_POSTINGS, index=False, encoding="utf-8-sig")
print(f"Exported: workday_postings.csv")
print(f"  Shape: ({len(df)}, {len(postings_cols)})")

# --- Export workday_training.csv (ML training set -- no raw_text!) ---
training_cols = [
    "censored_text", "compensation_grade", "job_profile_code",
    "job_family", "exempt_status", "pay_type"
]
df[training_cols].to_csv(OUT_TRAINING, index=False, encoding="utf-8-sig")
print(f"
Exported: workday_training.csv")
print(f"  Shape: ({len(df)}, {len(training_cols)})")
print(f"  Confirmed: 'raw_text' NOT in training CSV columns")

# --- Quick reload test ---
df_reload = pd.read_csv(OUT_TRAINING, encoding="utf-8-sig")
assert df_reload.shape[0] == len(df), "Row count mismatch after reload!"
assert "raw_text" not in df_reload.columns, "raw_text leaked into training CSV!"
print(f"
Reload test: PASS (shape={df_reload.shape})")


---
## 5. Validation

### Two-Tier Validation Approach

Validation uses two complementary strategies:

1. **Tier 1 -- Programmatic leakage scan:** Test every censored PD (all 103 rows)
   against 7 regex patterns that would indicate information leakage. This catches
   systematic failures (e.g., a regex that misses a pattern variant).

2. **Tier 2 -- Visual spot-check:** Select 15 random postings (seed=42 for
   reproducibility) and display the censored text alongside extracted labels.
   A human reviewer can verify that (a) the text reads naturally without gaps,
   and (b) no classification metadata is visible.

**Why both?** Programmatic scans catch what we know to look for (JP codes, comp
grades, dollar amounts). Visual spot-checks catch what we *don't* know -- unusual
formatting, orphaned punctuation, or semantic leakage that regex can't detect.

In [ ]:
# ===================================================================
# TIER 1: Comprehensive Programmatic Leakage Scan (ALL 103 rows)
# ===================================================================

training = pd.read_csv(OUT_TRAINING, encoding="utf-8-sig")
postings_full = pd.read_csv(OUT_POSTINGS, encoding="utf-8-sig")

leakage_patterns = [
    ("JP codes",              r"JP\d{3,6}"),
    ("Comp grade codes",      r"\b[SH]\d{1,2}\b"),
    ("Grade headers",         r"(?i)Compensation Grade|Pay Grade|Salary Grade"),
    ("Job Requisition IDs",   r"JR\d{5,6}"),
    ("Job Family headers",    r"Job Family:|Occupational Family:"),
    ("Dollar amounts",        r"\$\d{2,3},\d{3}"),
    ("Job Profile header",    r"Job Profile:"),
]

total_leaks = 0
results = []

print("=" * 60)
print("TIER 1: PROGRAMMATIC LEAKAGE SCAN")
print("=" * 60)

for pattern_name, regex in leakage_patterns:
    compiled = re.compile(regex)
    match_count = 0

    for idx, row in training.iterrows():
        text = str(row["censored_text"])
        found = compiled.findall(text)
        if found:
            # Filter false positives for comp grade codes
            if pattern_name == "Comp grade codes":
                real = []
                for m in compiled.finditer(text):
                    before = text[m.start()-1] if m.start() > 0 else " "
                    after = text[m.end()] if m.end() < len(text) else " "
                    if not before.isalpha() and not after.isalpha():
                        real.append(m.group())
                match_count += len(real)
            else:
                match_count += len(found)

    total_leaks += match_count
    status = "CLEAN" if match_count == 0 else f"LEAK ({match_count})"
    results.append((pattern_name, match_count, status))
    print(f"  {pattern_name:<25s} {match_count:>3d} matches  [{status}]")

print(f"\nTier 1 summary: {len(leakage_patterns)} patterns x {len(training)} rows")
print(f"Total leaks: {total_leaks}")
print(f"Tier 1 verdict: {'PASS' if total_leaks == 0 else 'FAIL'}")

In [ ]:
# ===================================================================
# TIER 2: Visual Spot-Check (15 random samples)
# ===================================================================

random.seed(42)
sample_indices = sorted(random.sample(range(len(postings_full)), 15))

print("=" * 60)
print("TIER 2: VISUAL SPOT-CHECK (15 random samples, seed=42)")
print("=" * 60)

for i, idx in enumerate(sample_indices, 1):
    row = postings_full.iloc[idx]
    title = str(row.get("title", "N/A"))
    comp = str(row.get("compensation_grade", "N/A"))
    jp = str(row.get("job_profile_code", "N/A"))
    jf = str(row.get("job_family", "N/A"))
    ex = str(row.get("exempt_status", "N/A"))
    censored = str(row.get("censored_text", ""))[:400]

    print(f"\n--- Spot-check {i}/15: Row {idx} ---")
    print(f"  Title:    {title}")
    print(f"  Grade={comp}  JP={jp}  Family={jf}  Exempt={ex}")
    print(f"  Censored (first 400 chars):")
    print(f"  {censored}...")

### Validation Summary

| Check | Result |
|-------|--------|
| Total postings validated | 103 |
| Leakage patterns tested | 7 |
| Total leaks found | 0 |
| Visual spot-checks | 15 (seed=42) |
| Overall verdict | **PASS** |

The censored training data is clean: no JP codes, no compensation grades, no
dollar amounts, no job family headers, and no JR IDs leaked into the text that
the ML model will see.

---
## 6. What's Next

With 103 labeled, censored position descriptions in `workday_training.csv`,
the GRIFFIN pipeline continues:

| Next Step | Notebook | What it does |
|-----------|----------|--------------|
| **A4: Feature Engineering** | `griffin_feature_engineering.ipynb` | Extract structured signals from censored PDs: text length, keyword flags, n-gram features, TF-IDF vectors. Output: tabular feature matrix for ML. |
| **A5: H2O AutoML Training** | `griffin_h2o_training.ipynb` | Train classification models (GBM, XGBoost, DNN) to predict occupational family from tabular features. Evaluate with cross-validation and SHAP explanations. |
| **B1: LangChain Multi-Agent** | `griffin_langchain_agent.ipynb` | Wire up the three-stage architecture: ML Shortlist (H2O top-3) -> LLM Classifier (Gemini/Claude reasoning) -> Confidence Check (threshold + escalation). |

### Quick reference: output files from this notebook

| File | Rows | Columns | Use |
|------|------|---------|-----|
| `workday_postings.csv` | 103 | 12 | Master dataset with raw + censored text + all labels |
| `workday_training.csv` | 103 | 6 | ML training input (censored text + labels only) |
| `workday_exclusions.csv` | 47 | 4 | Audit log of faculty exclusions |
| `raw_workday/` | -- | -- | Cached API responses (list pages + detail JSONs) |